In [33]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

In [9]:
# load file with private area
erf_gpd = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/Erf_frame.geojson")
erf = erf_gpd[["gml_id", "eindRegistratie", "bgt-fysiekVoorkomen", "geometry"]]
erf = erf[erf["eindRegistratie"].isna()]

In [14]:
# Load file with buildings
bld_gpd = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/buildings_osm_frame.gpkg")
bld = bld_gpd[["osm_id", "type", "geometry"]]

In [36]:
# Load file with neigbourhood border
buurt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_BUURT (1).csv", delimiter=';')[["Buurt", "Wijk", "WKT_LNG_LAT"]]
buurt['geometry'] = buurt['WKT_LNG_LAT'].apply(wkt.loads)
buurt_gdf = gpd.GeoDataFrame(buurt, geometry='geometry', crs="EPSG:4326")
buurt_gdf = buurt_gdf.to_crs("EPSG:28992")

In [21]:
# Merge buildings and private property into a single private space layer
gdf_private_space = gpd.GeoDataFrame(pd.concat([bld, erf], ignore_index=True))

# Dissolve to get total private space as a single geometry, also to not have overlaying areas
gdf_private_space = gdf_private_space.dissolve()

In [44]:
# Dissolve to get total erf as a single geometry
gdf_erf = erf.dissolve()

In [46]:
# Overlay to connect erf space to neighbourhood
gdf_erf_by_neighborhood = gpd.overlay(gdf_erf, buurt_gdf, how = 'intersection')

In [40]:
# Overlay to connect private space to neighbourhood
gdf_public_by_neighborhood = gpd.overlay(gdf_private_space, buurt_gdf, how='intersection')

In [48]:
#Only erf by neighbourhood
gdf_erf_by_neighborhood.to_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/neighbourhood_erf.gpkg", driver="GPKG")

In [42]:
#Private space by neighbourhood
gdf_public_by_neighborhood.to_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/neighbourhood_private_space.gpkg", driver="GPKG")

In [29]:
#whole private space with buildings as one multipolygon
gdf_private_space.to_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/private_space.gpkg", driver="GPKG")

In [26]:
erf.to_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/erf_space.geojson", driver="GeoJSON")